In [2]:
# Resample/Cropping Code
# Required:
#    MTLE root directory - withsegmentation, anatomical and temperature pngs files + caseNotes CSV
#    segmentationData directory - defined by user in main function

Main function is located at the end of the notebook. It loops 

import SimpleITK as sitk
import numpy as np
import matplotlib.pyplot as plt
import pydicom
import glob
import os
import shutil
import random
import pydicom
import pandas as pd
import math
from PIL import Image

In [4]:
# For each patient
    # For each acquisition
        # Get all S2 data in one folder

        # Read temperatureDataS2 dicom series as SITK image
        # Read anatomicalData dicom series as SITK image

        # Create resampler 

        # Acquire laser location from csv and turn into world coordinates
        # Project laser location onto segmentation image

        # Crop and save segmentation

In [5]:
# Isolate S2 slices from temperatureData
def getS2(source_folder):
    destination_folder = source_folder.replace("temperatureData", "temperatureDataS2")
    if not os.path.exists(destination_folder):
        os.makedirs(destination_folder)

    for root, dirs, files in os.walk(source_folder):
        for file in files:
            if 'S2' in file:
                source_path = os.path.join(root, file)
                destination_path = os.path.join(destination_folder, file)
                shutil.copy(source_path, destination_path)


In [6]:
# Load in dicom series as SITK image
def SITK_loader(filepath):
    reader = sitk.ImageSeriesReader()
    dicom_names = reader.GetGDCMSeriesFileNames(filepath)
    reader.SetFileNames(dicom_names)
    sitk_img = reader.Execute()
    
    return sitk_img

In [8]:
# Resampler from anatomicalSeg to temperatureDataS2
def create_resampler(source_sitk, target_sitk):

    resampler = sitk.ResampleImageFilter()
    resampler.SetReferenceImage(target_sitk)
    resampler.SetDefaultPixelValue(source_sitk.GetPixelIDValue())
    resampler.SetSize((128, 128, 1))
    resampler.SetOutputSpacing((2.0,2.0,1.0))
    resampler.SetOutputOrigin(target_sitk.GetOrigin())
    resampler.SetOutputDirection(target_sitk.GetDirection())
    resampler.SetInterpolator(sitk.sitkLinear)
    resampled_img = resampler.Execute(source_sitk)
    resampled_img_array = sitk.GetArrayFromImage(resampled_img[:, :,0])

    # Combine GM and WM labels
    resampled_img_array[resampled_img_array == 3] = 2

    return resampled_img_array, resampled_img

SITK resampler reference:
http://insightsoftwareconsortium.github.io/SimpleITK-Notebooks/Python_html/21_Transforms_and_Resampling.html 
https://simpleitk.org/doxygen/latest/html/classitk_1_1simple_1_1ResampleImageFilter.html

In [9]:
# Obtain laser location from csv file
def get_laser(filepath):
    
    # read the CSV file
    df = pd.read_csv(filepath)
    
    #Find the probe location from pandas dataframe 
    laser_loc_x = float(df.iloc[10][1])
    laser_loc_y = float(df.iloc[11][1])
    laser_loc_z = float(df.iloc[12][1])
    
    return [laser_loc_x, laser_loc_y, laser_loc_z]

In [11]:
# Project dicom coordinate laser location into world coordinates
def project_laser(resampled_img, laser_loc):
    return resampled_img.TransformPhysicalPointToContinuousIndex((laser_loc))

In [12]:
# Crop image to 51x51 using specified centre location
def Crop(center_point, img_array):

    # Adjust box size if needed
    boxSize = 51
    laser = np.array(center_point)

    # xPos and yPox and upper left position of cropped window
    xPosition = round(laser[0] - boxSize/2)
    yPosition = round(laser[1] - boxSize/2)

    # Compensate for one pixel shift
    if round(laser[0]) == 63 and round(laser[1]) == 64: # If laser is shifted one down
        # Shift one up
        xPosition += 1
    if round(laser[0]) == 64 and round(laser[1]) == 63: # If laser is shifted one right
        # Shift one left
        yPosition += 1

    # Crop the image
    cropped_img = img_array[xPosition:xPosition+boxSize, yPosition:yPosition+boxSize]

    return cropped_img



Extra Explanation for pixel shift:
- There wasn't a concrete way to fix the one pixel shift between images so I tried to find a pattern between the computed x,y laser position and how it was incorrectly shited
- The code I used "SEEMED" to work, and I tested it on as many images as possible
- This is just a warning incase you later notice that there is a one-pixel shift anywhere

In [14]:
# Save segmentation array as png
def saveImage(img_array, save_dir):
    
    # Normalize pixel values to 0-255
    segmentation_cropped_normalized = (img_array - np.min(img_array)) / (np.max(img_array) - np.min(img_array)) * 255
    segmentation_cropped_normalized = segmentation_cropped_normalized.astype(np.uint8)

    # Convert NumPy array to PIL Image
    segmentation_cropped_image = Image.fromarray(segmentation_cropped_normalized)
    print(segmentation_cropped_image.size)

    # Save the cropped image as PNG
    segmentation_cropped_image.save(save_dir)

In [1]:
## MAIN FUNCTION

root_directory = "E:\Documents\\MTLE"
save_png_dir = "E:\Documents\MRgLITT\data\\500J\original\segmentationData"
target_name = (os.path.join(root_directory, '**','**', ))
acquisitions = glob.glob(target_name)

# Number of acquisitions in MTLE file
num = len(acquisitions)
for ac in acquisitions:

    print("Currently working on:", ac)

    # Get CaseNotes Csv File
    caseNotes_dir = os.path.join(ac, "caseNotes.csv")

    # Create seg SITK image
    sitkSeg_dir = os.path.join(ac, "anatomicalSeg")
    seg_sitk = SITK_loader(sitkSeg_dir)

    # Copy only S2 dcms to new folder
    all_temp = os.path.join(ac, "temperatureData")
    getS2(all_temp)

    # Create S2 SITK image
    sitkTemp_dir = os.path.join(ac, "temperatureDataS2")
    temp_sitk = SITK_loader(sitkTemp_dir)

    # Create resampled segmentation image
    resampled_segmentation, resampled_sitk = create_resampler(seg_sitk, temp_sitk)

    # Get laser location of current acquisition
    laser_loc = get_laser(caseNotes_dir)
    
    # Get Projected Laser
    projected_laser = project_laser(resampled_sitk, laser_loc)

    # Crop to 51x51 with center point as laser location
    cropped_seg = Crop(projected_laser, resampled_segmentation)
    # cropped_seg = Crop([63.5,63.5], resampled_segmentation)

    # # Save segmentationa as PNG
    save_dir = os.path.join(save_png_dir, os.path.basename(ac)) + '-S2CroppedSeg.png'
    saveImage(cropped_seg,save_dir)
    break #REMOVE

    
        



In [2]:
# ## Tester - plot anatomicalProbesEye png + segmentationData png
# from matplotlib.colors import LinearSegmentedColormap
# from matplotlib.colors import ListedColormap
# colors_1=["white", "darkmagenta","blue","green","yellow","orange","red", "black"]
# nodes_1 = [0.0, 0.04, 0.078, 0.14, 0.17, 0.23, 0.34, 1.0]
# Monteris_cmap = LinearSegmentedColormap.from_list("mycmap", list(zip(nodes_1, colors_1)))

# colors = ['black', 'purple', 'orange']
# integer_values = [0, 1, 2]
# labels = ['CSF', 'Brain Matter']

# # Create the custom colormap
# custom_colormap = ListedColormap(colors)

# # probes = Image.open("E:\Documents\MRgLITT\data\\500J\\first_acquisition\\temperatureData\LP-0002-01-01-01-089.S2Cropped.png")
# # probes = Image.open("E:\Documents\MRgLITT\data\\500J\\first_acquisition\\\LP-0002-01-01-01-065.S2Cropped.png")
# probes = Image.open("E:\Documents\MRgLITT\data\\500J\\first_acquisition\\anatomicalProbesEye\\\LP-0002-01-01-01-S2Cropped.png")

# seg = Image.open("E:\Documents\MRgLITT\data\\500J\original\segmentationData\LP-0002-01-01-01-S2CroppedSeg.png")

# plt.title("Overlay of LP-0002-01-01-01")
# plt.imshow(probes, cmap = 'hot')
# plt.imshow(seg,cmap = custom_colormap, vmin=0, vmax=255,alpha = 0.1)
# plt.scatter(25,25, color = 'red', s=10)
